# Video branch: screen text (OCR) from the Spoken Tutorial videos

Runtime → Change runtime type → **T4 GPU**, then run the cells in order. Every stage saves to Drive and skips finished work, so re-running after a disconnect is safe. **No manual video download is needed**: stage 0 downloads all 25 Hindi videos (old versions matching MUCS) from their direct links.

| Stage | What it does | Time |
|---|---|---|
| 0 | **Download** the 25 videos (~250 MB) into `videos/` | ~5 min |
| 1 | Download the MUCS test set again (needed only to line up audio) | ~3 min |
| 2 | **Align**: find where each window starts/ends in its video by matching the audio; also confirms each video is the right one | ~5 min |
| 3 | **OCR**: sample frames, keep frames where the screen changes, read the English screen text | ~20-30 min |
| 4 | **Build** the new inputs (transcript + screen text; screen text only) | seconds |
| 5 | **Summarise** with the same Qwen setup: 100 summaries | ~30 min |

After stage 2, look at the printed table: `conf_first` and `conf_last` should be well above 3 and `method` should be `audio_xcorr`. Any `proportional_fallback` row means that video may not match its window; tell me which. At the end, send `summaries_video.csv`, `screen_text.csv` and `alignment.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/hinglish_asr_sum'
%cd {ROOT}
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pytesseract imagehash bitsandbytes accelerate
!tesseract --version | head -1

In [ ]:
%%writefile video_urls.csv
window_id,watch_url,video_url,alt_video_url
W00,https://spoken-tutorial.org/watch/Thunderbird/How+to+Use+Thunderbird/Hindi,https://spoken-tutorial.org/media/videos/34/320/How-to-Use-Thunderbird-Hindi.webm,
W01,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Slide+Master+Slide+Design/Hindi,https://spoken-tutorial.org/media/videos/18/329/Slide-Master-Slide-Design-Hindi.webm,
W02,https://spoken-tutorial.org/watch/gedit+Text+Editor/Introduction+to+gedit+Text+Editor/Hindi,https://spoken-tutorial.org/media/videos/84/809/Introduction-to-gedit-Text-Editor-Hindi.webm,
W03,https://spoken-tutorial.org/watch/gedit+Text+Editor/Snippets+in+gedit/Hindi,https://spoken-tutorial.org/media/videos/84/843/Snippets-in-gedit-Hindi.webm,
W04,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Custom+Animation/Hindi,https://spoken-tutorial.org/media/videos/18/330/Custom-Animation-Hindi.webm,
W05,https://spoken-tutorial.org/watch/Thunderbird/Introduction+to+Thunderbird/Hindi,https://spoken-tutorial.org/media/videos/34/319/Introduction-to-Thunderbird-Hindi.webm,
W06,https://spoken-tutorial.org/watch/Advance+C/Command+line+arguments+in+C/Hindi,https://spoken-tutorial.org/media/videos/61/605/Command-line-arguments-in-C-Hindi.webm,
W07,https://spoken-tutorial.org/watch/LibreOffice+Suite+Writer/Using+track+changes/Hindi,https://spoken-tutorial.org/media/videos/20/252/Using-track-changes-Hindi.webm,
W08,https://spoken-tutorial.org/watch/LibreOffice+Suite+Writer/Viewing+and+printing+a+text+document/Hindi,https://spoken-tutorial.org/media/videos/20/39/Viewing-and-printing-a-text-document-Hindi.webm,
W09,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Creating+a+presentation+document/Hindi,https://spoken-tutorial.org/media/videos/18/57/Creating-a-presentation-document-Hindi.webm,
W10,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Introduction+to+LibreOffice+Impress/Hindi,https://spoken-tutorial.org/media/videos/18/56/Introduction-to-LibreOffice-Impress-Hindi.webm,
W11,https://spoken-tutorial.org/watch/JChemPaint/Properties+of+JChemPaint/Hindi,https://spoken-tutorial.org/media/videos/59/602/Properties-of-JChemPaint-Hindi.webm,
W12,https://spoken-tutorial.org/watch/Advance+C/Union+and+Typedef/Hindi,https://spoken-tutorial.org/media/videos/61/604/Union-and-Typedef-Hindi.webm,
W13,https://spoken-tutorial.org/watch/JChemPaint/Introduction+to+JChemPaint/Hindi,https://spoken-tutorial.org/media/videos/59/586/Introduction-to-JChemPaint-Hindi.webm,
W14,https://spoken-tutorial.org/watch/Thunderbird/Account+settings+and+configuring/Hindi,https://spoken-tutorial.org/media/videos/34/322/Account-settings-and-configuring-Hindi.webm,
W15,https://spoken-tutorial.org/watch/LibreOffice+Suite+Writer/Typing+text+and+basic+formatting/Hindi,https://spoken-tutorial.org/media/videos/20/228/Typing-text-and-basic-formatting-Hindi.webm,
W16,https://spoken-tutorial.org/watch/Advance+C/Storage+class+specifiers/Hindi,https://spoken-tutorial.org/media/videos/61/606/Storage-class-specifiers-Hindi.webm,
W17,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Presentation+Notes/Hindi,https://spoken-tutorial.org/media/videos/18/332/Presentation-Notes-Hindi.webm,
W18,https://spoken-tutorial.org/watch/LibreOffice+Suite+Writer/Using+search+replace+auto+correct/Hindi,https://spoken-tutorial.org/media/videos/20/249/Using-search-replace-auto-correct-Hindi.webm,
W19,https://spoken-tutorial.org/watch/LibreOffice+Suite+Writer/Creating+Newsletter/Hindi,https://spoken-tutorial.org/media/videos/20/40/Creating-newsletter-Hindi.webm,
W20,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Slide+Creation/Hindi,https://spoken-tutorial.org/media/videos/18/331/Slide-Creation-Hindi.webm,
W21,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Printing+a+Presentation+Document/Hindi,https://spoken-tutorial.org/media/videos/18/60/Printing-a-Presentation-Document-Hindi.webm,
W22,https://spoken-tutorial.org/watch/JChemPaint/Features+of+JChemPaint/Hindi,https://spoken-tutorial.org/media/videos/59/601/Features-of-JChemPaint-Hindi.webm,
W23,https://spoken-tutorial.org/watch/LibreOffice+Suite+Impress/Viewing+a+Presentation+Document/Hindi,https://spoken-tutorial.org/media/videos/18/58/Viewing-a-Presentation-Document-Hindi.webm,
W24,https://spoken-tutorial.org/watch/Xfig/Feedback+control+diagram/Hindi,https://spoken-tutorial.org/media/videos/37/1117/Feedback-control-diagram-Hindi.webm,https://spoken-tutorial.org/media/videos/37/1118/Feedback-diagram-with-Maths-Hindi.webm


In [ ]:
%%writefile video_pipeline.py
"""Video branch: align each window to its Spoken Tutorial video, OCR the screen, build inputs.

Usage (Colab, from the Drive project folder):
    python video_pipeline.py download --root ROOT          (reads ROOT/video_urls.csv)
    python video_pipeline.py align   --root ROOT --mucs /content/mucs_data/raw/extracted
    python video_pipeline.py ocr     --root ROOT
    python video_pipeline.py build   --root ROOT

Inputs:  ROOT/videos/W00.<ext> ... W24.<ext>   (Hindi Spoken Tutorial videos, any ffmpeg-readable format)
         ROOT/outputs/selected_windows.csv, ROOT/outputs/step2/conditions.csv
Outputs (all new files under ROOT/outputs/video/):
  alignment.csv        window -> [t_start, t_end] in the video, how it was found, confidence
  frames/W00/*.png     kept (de-duplicated) frames
  ocr_lines.csv        every kept OCR line: window, time (s from window start), text, confidence
  screen_text.csv      one row per window: the de-duplicated screen text in time order
  conditions_video.csv inputs for the summariser (transcript + screen text, and screen text only)
"""
import argparse, glob, json, os, re, subprocess, sys
import numpy as np
import pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument("stage", choices=["download", "align", "ocr", "build"])
ap.add_argument("--root", required=True)
ap.add_argument("--mucs", help="extracted MUCS test folder (needed for align)")
ap.add_argument("--fps", type=float, default=0.5, help="frames per second sampled before de-duplication")
ap.add_argument("--max-screen-chars", type=int, default=6000)
A = ap.parse_args()
ROOT = A.root
OUT = f"{ROOT}/outputs/video"
os.makedirs(OUT, exist_ok=True)
SR = 8000
sel = pd.read_csv(f"{ROOT}/outputs/selected_windows.csv")
WINS = list(sel.window_id)


def video_path(w):
    c = sorted(p for p in glob.glob(f"{ROOT}/videos/{w}.*") if not p.endswith(".part"))
    return c[0] if c else None


def video_audio(path):
    """Mono 8 kHz float32 audio of the video's soundtrack via ffmpeg."""
    raw = subprocess.run(["ffmpeg", "-v", "error", "-i", path, "-ac", "1", "-ar", str(SR), "-f", "f32le", "-"],
                         capture_output=True, check=True).stdout
    return np.frombuffer(raw, dtype=np.float32).copy()


def video_duration(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", path],
                         capture_output=True, text=True).stdout.strip()
    return float(out)


def locate(template, signal):
    """Position (s) of template inside signal by normalised FFT cross-correlation, plus a confidence:
    peak height divided by the 99.9th percentile of the correlation (>3 = clear match)."""
    from scipy.signal import fftconvolve
    t = (template - template.mean()) / (template.std() + 1e-8)
    s = signal - signal.mean()
    corr = fftconvolve(s, t[::-1], mode="valid")
    # local energy normalisation so loud passages do not win by default
    win = np.ones(len(t))
    energy = np.sqrt(np.maximum(fftconvolve(s ** 2, win, mode="valid"), 1e-8))
    ncc = corr / energy
    k = int(np.argmax(ncc))
    conf = float(ncc[k] / (np.percentile(np.abs(ncc), 99.9) + 1e-8))
    return k / SR, conf



# --------------------------------------------------------------------------------------------- download
if A.stage == "download":
    urls = pd.read_csv(f"{ROOT}/video_urls.csv", keep_default_na=False)
    os.makedirs(f"{ROOT}/videos", exist_ok=True)
    for r in urls.itertuples():
        if video_path(r.window_id):
            print(r.window_id, "already downloaded"); continue
        ext = os.path.splitext(r.video_url)[1] or ".webm"
        dst = f"{ROOT}/videos/{r.window_id}{ext}"
        rc = subprocess.run(["wget", "-q", "--tries=3", "--timeout=60", "-O", dst + ".part", r.video_url]).returncode
        if rc == 0 and os.path.getsize(dst + ".part") > 200_000:
            os.rename(dst + ".part", dst)
            print(r.window_id, "ok", round(os.path.getsize(dst) / 1e6, 1), "MB", round(video_duration(dst) / 60, 1), "min")
        else:
            print(r.window_id, "FAILED - download it in a browser from", r.watch_url, "and save as", os.path.basename(dst))
            if os.path.exists(dst + ".part"):
                os.remove(dst + ".part")

# --------------------------------------------------------------------------------------------- align
if A.stage == "align":
    import soundfile as sf
    from scipy.signal import resample_poly
    seg_files = glob.glob(f"{A.mucs}/**/segments", recursive=True)
    scp_files = glob.glob(f"{A.mucs}/**/wav.scp", recursive=True)
    assert seg_files, "MUCS segments file not found under --mucs"
    kdir = os.path.dirname(seg_files[0])
    seg = pd.read_csv(seg_files[0], sep=r"\s+", header=None, names=["utt", "reco", "start", "end"])
    wavs = {os.path.basename(p): p for p in glob.glob(f"{A.mucs}/**/*.wav", recursive=True)}
    scp = {}
    if scp_files:
        for line in open(scp_files[0]):
            k, v = line.strip().split(None, 1)
            cand = [x for x in v.split() if x.endswith(".wav")] or [v]
            p = cand[0]
            scp[k] = p if os.path.isfile(p) else (os.path.join(kdir, p) if os.path.isfile(os.path.join(kdir, p))
                                                  else wavs.get(os.path.basename(p)))

    def mucs_span(reco, start, end):
        path = scp.get(reco) or wavs.get(reco + ".wav")
        info = sf.info(path)
        x, sr = sf.read(path, start=int(start * info.samplerate), stop=int(end * info.samplerate),
                        dtype="float32", always_2d=True)
        x = x.mean(axis=1)
        g = np.gcd(int(sr), SR)
        return resample_poly(x, SR // g, int(sr) // g).astype("float32")

    rows = []
    for w in sel.itertuples():
        vp = video_path(w.window_id)
        if not vp:
            print(f"{w.window_id}: video missing"); continue
        va = video_audio(vp)
        ids = w.utt_ids.split()
        s = seg.set_index("utt")
        first, last = s.loc[ids[0]], s.loc[ids[-1]]
        # use up to 6 s of the first and last utterance as templates
        f_t = mucs_span(w.reco_id, first.start, min(first.end, first.start + 6))
        l_t = mucs_span(w.reco_id, max(last.start, last.end - 6), last.end)
        t_first, c_first = locate(f_t, va)
        t_last, c_last = locate(l_t, va)
        t_start, t_end = t_first, t_last + len(l_t) / SR
        ok = min(c_first, c_last) > 3 and t_end > t_start
        method = "audio_xcorr"
        if not ok:   # fallback: proportional position of the window's speech inside the recording's speech
            tot = seg[seg.reco == w.reco_id]
            frac = (last.end - tot.start.min()) / (tot.end.max() - tot.start.min())
            t_start, t_end, method = 0.0, frac * video_duration(vp), "proportional_fallback"
        rows.append(dict(window_id=w.window_id, video=os.path.basename(vp), t_start=round(t_start, 2),
                         t_end=round(t_end, 2), conf_first=round(c_first, 2), conf_last=round(c_last, 2),
                         method=method, video_sec=round(len(va) / SR, 1), window_speech_sec=w.speech_sec))
        print(rows[-1], flush=True)
    al = pd.DataFrame(rows)
    al.to_csv(f"{OUT}/alignment.csv", index=False)
    print(al.method.value_counts().to_string())
    print("Check: wall-clock span should be a bit longer than speech_sec (pauses):")
    print((al.t_end - al.t_start - al.window_speech_sec).describe().round(1).to_string())

# --------------------------------------------------------------------------------------------- ocr
if A.stage == "ocr":
    import pytesseract, imagehash
    from PIL import Image
    al = pd.read_csv(f"{OUT}/alignment.csv")
    done = set(pd.read_csv(f"{OUT}/ocr_lines.csv").window_id) if os.path.exists(f"{OUT}/ocr_lines.csv") else set()
    for r in al.itertuples():
        if r.window_id in done:
            continue
        fdir = f"{OUT}/frames/{r.window_id}"
        os.makedirs(fdir, exist_ok=True)
        tmp = f"/content/frames_tmp_{r.window_id}"
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(["ffmpeg", "-v", "error", "-ss", str(r.t_start), "-to", str(r.t_end), "-i",
                        f"{ROOT}/videos/{r.video}", "-vf", f"fps={A.fps}", f"{tmp}/f_%05d.png"], check=True)
        frames = sorted(glob.glob(f"{tmp}/f_*.png"))
        kept, prev = [], None
        for i, fp in enumerate(frames):
            h = imagehash.phash(Image.open(fp))
            if prev is None or h - prev > 6:          # screen changed noticeably
                kept.append((i / A.fps, fp)); prev = h
        seen, lines = set(), []
        for t, fp in kept:
            img = Image.open(fp).convert("L")
            img = img.resize((img.width * 2, img.height * 2))       # small UI fonts OCR better upscaled
            d = pytesseract.image_to_data(img, lang="eng", config="--psm 11", output_type=pytesseract.Output.DATAFRAME)
            d = d[(d.conf > 60) & d.text.notna()]
            d["text"] = d.text.astype(str).str.strip()
            d = d[d.text.str.len() > 0]
            for key, g in d.groupby(["block_num", "par_num", "line_num"]):
                txt = " ".join(g.text)
                norm = re.sub(r"[^a-z0-9+.>/_\- ]", "", txt.lower()).strip()
                letters = sum(ch.isalpha() for ch in norm)
                if letters < 3 or norm in seen:
                    continue
                seen.add(norm)
                lines.append(dict(window_id=r.window_id, t=round(t, 1), text=txt, conf=round(g.conf.mean(), 1)))
            Image.open(fp).save(f"{fdir}/t{int(t):04d}.png")
        for fp in frames:
            os.remove(fp)
        pd.DataFrame(lines).to_csv(f"{OUT}/ocr_lines.csv", mode="a", header=not os.path.exists(f"{OUT}/ocr_lines.csv"),
                                   index=False)
        print(f"{r.window_id}: {len(frames)} frames sampled, {len(kept)} kept, {len(lines)} new text lines", flush=True)

# --------------------------------------------------------------------------------------------- build
if A.stage == "build":
    ocr = pd.read_csv(f"{OUT}/ocr_lines.csv", keep_default_na=False)
    cond = pd.read_csv(f"{ROOT}/outputs/step2/conditions.csv", keep_default_na=False)
    st_rows, rows = [], []
    for w in WINS:
        g = ocr[ocr.window_id == w].sort_values("t")
        text = "\n".join(f"[{int(t // 60):02d}:{int(t % 60):02d}] {x}" for t, x in zip(g.t, g.text))
        if len(text) > A.max_screen_chars:
            text = text[:A.max_screen_chars].rsplit("\n", 1)[0]
        st_rows.append(dict(window_id=w, n_lines=len(g), screen_text=text, chars=len(text)))
        for c in ["oracle", "asr:trelis-hinglish-preview", "asr:whisper-small"]:
            tr = cond[(cond.window_id == w) & (cond.condition == c)].text.iloc[0]
            rows.append(dict(window_id=w, condition=f"{c}+screen", transcript=tr, screen_text=text))
        rows.append(dict(window_id=w, condition="screen_only", transcript="", screen_text=text))
    pd.DataFrame(st_rows).to_csv(f"{OUT}/screen_text.csv", index=False)
    pd.DataFrame(rows).to_csv(f"{OUT}/conditions_video.csv", index=False)
    print(pd.DataFrame(st_rows)[["window_id", "n_lines", "chars"]].to_string(index=False))


In [ ]:
%%writefile run_qwen_video.py
"""Summarise the video conditions with the same Qwen setup as Step 2 (resumable).
    python run_qwen_video.py --root ROOT
Reads outputs/video/conditions_video.csv; writes outputs/video/summaries_video.csv.
Settings identical to Step 2 (Qwen2.5-7B-Instruct @ a09a354, nf4 4-bit, greedy, batch 4, max_new_tokens 320).
The transcript+screen prompt is the Step-2 summary_en prompt with one added sentence and a screen-text block;
the screen-only prompt describes OCR text instead of a transcript. Both are logged to prompts_log.jsonl."""
import argparse, datetime as dt, glob, hashlib, json, os
import pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ap = argparse.ArgumentParser(); ap.add_argument("--root", required=True); A = ap.parse_args()
ROOT = A.root; OUT = f"{ROOT}/outputs/video"; CACHE = f"{ROOT}/cache/qwen_video"; os.makedirs(CACHE, exist_ok=True)
MODEL, REV, BATCH = "Qwen/Qwen2.5-7B-Instruct", "a09a35458c702b33eeacc393d103063234e8bc28", 4
GEN = dict(max_new_tokens=320, do_sample=False)

logged = [json.loads(l) for l in open(f"{ROOT}/logs/prompts_log.jsonl")]
base = [d["prompt"] for d in logged if d.get("component") == "summary_en" and d.get("step") == "2-templates"][0]
SYSTEM, BASE_T = base["system"], base["user_template"]
head, tail = BASE_T.split("\n\nTranscript:\n")            # tail = "<<<\n{transcript}\n>>>"
assert tail == "<<<\n{transcript}\n>>>", "unexpected Step-2 template"
SCREEN_T = (head + "\n\nAfter the transcript you also get the text visible on screen in the tutorial video, read "
            "by OCR in time order; it may contain menu names, commands and recognition errors.\n\nTranscript:\n"
            + tail + "\n\nScreen text:\n<<<\n{screen}\n>>>")
ONLY_T = ("The following is the text visible on screen in a recorded computer-software tutorial video, read by OCR "
          "in time order; it may contain menu names, commands and recognition errors. There is no transcript.\n\n"
          "Write a summary in English of 120-180 words covering the main steps, concepts and facts taught. "
          "Write plain prose: no bullet points, no title.\n\nScreen text:\n<<<\n{screen}\n>>>")

cv = pd.read_csv(f"{OUT}/conditions_video.csv", keep_default_na=False)
def prompt(r):
    u = ONLY_T.format(screen=r.screen_text) if r.condition == "screen_only" else \
        SCREEN_T.format(transcript=r.transcript, screen=r.screen_text)
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": u}]
key = lambda w, c: hashlib.sha256(f"{w}|{c}|video".encode()).hexdigest()[:24]
todo = [r for r in cv.itertuples() if not os.path.exists(f"{CACHE}/{key(r.window_id, r.condition)}.json")]
print(f"{len(cv) - len(todo)} cached, {len(todo)} to generate")
if todo:
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REV); tok.padding_side = "left"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, quantization_config=bnb, device_map="auto",
                                                 dtype=torch.float16).eval()
    for i in range(0, len(todo), BATCH):
        b = todo[i:i + BATCH]
        enc = tok([tok.apply_chat_template(prompt(r), tokenize=False, add_generation_prompt=True) for r in b],
                  return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, **GEN, pad_token_id=tok.eos_token_id)
        for k, r in enumerate(b):
            txt = tok.decode(out[k][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            json.dump(dict(created=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), model=MODEL,
                           revision=REV, gen=str(GEN), batch=BATCH, window_id=r.window_id, condition=r.condition,
                           out_lang="en", input_tokens=int(enc["attention_mask"][k].sum()), summary=txt),
                      open(f"{CACHE}/{key(r.window_id, r.condition)}.json", "w"), ensure_ascii=False)
        print(f"done {min(i + BATCH, len(todo))}/{len(todo)}", flush=True)
    with open(f"{ROOT}/logs/prompts_log.jsonl", "a") as f:
        for comp, t in [("summary_en_screen", SCREEN_T), ("summary_en_screen_only", ONLY_T)]:
            f.write(json.dumps({"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "step": "3-video",
                                "component": comp, "model": MODEL, "params": dict(revision=REV, **GEN, batch=BATCH),
                                "prompt": {"system": SYSTEM, "user_template": t}}, ensure_ascii=False) + "\n")
df = pd.DataFrame([json.load(open(p)) for p in glob.glob(f"{CACHE}/*.json")])
df["summary_words"] = df.summary.str.split().str.len()
df["ends_clean"] = df.summary.str.strip().str[-1].isin(list('.!?"\')'))
df.sort_values(["condition", "window_id"]).to_csv(f"{OUT}/summaries_video.csv", index=False)
print(df.groupby("condition")[["summary_words", "input_tokens", "ends_clean"]].mean().round(1))


## Stage 0: download the videos

In [ ]:
!python video_pipeline.py download --root {ROOT}

## Stage 1: MUCS test set (for audio alignment only)

In [ ]:
import os, subprocess, tarfile
RAW = '/content/mucs_data/raw'; os.makedirs(RAW, exist_ok=True)
TAR = f'{RAW}/Hindi-English_test.tar.gz'; EXT = f'{RAW}/extracted'
URLS = ["https://openslr.trmal.net/resources/104/Hindi-English_test.tar.gz",
        "https://openslr.elda.org/resources/104/Hindi-English_test.tar.gz",
        "https://openslr.magicdatatech.com/resources/104/Hindi-English_test.tar.gz"]
if not os.path.exists(TAR):
    for u in URLS:
        if subprocess.run(['wget', '-q', '-O', TAR + '.part', u]).returncode == 0 and os.path.getsize(TAR + '.part') > 1e8:
            os.rename(TAR + '.part', TAR); break
if not os.path.exists(f'{EXT}/.done'):
    os.makedirs(EXT, exist_ok=True)
    with tarfile.open(TAR) as tf: tf.extractall(EXT)
    open(f'{EXT}/.done', 'w').close()
print('MUCS ready:', os.path.exists(f'{EXT}/.done'))

## Stage 2: align windows to videos

In [ ]:
!python video_pipeline.py align --root {ROOT} --mucs /content/mucs_data/raw/extracted

## Stage 3: OCR the screen

In [ ]:
!python video_pipeline.py ocr --root {ROOT}

In [ ]:
# Look at a few kept frames and the text read from them (sanity check)
import pandas as pd, glob
from IPython.display import display, Image as IPImage
o = pd.read_csv(f'{ROOT}/outputs/video/ocr_lines.csv', keep_default_na=False)
print(o.groupby('window_id').size().describe())
print(o[o.window_id == 'W00'].head(25).to_string(index=False))
for p in sorted(glob.glob(f'{ROOT}/outputs/video/frames/W00/*.png'))[:3]:
    display(IPImage(p, width=480))

## Stage 4: build inputs

In [ ]:
!python video_pipeline.py build --root {ROOT}

## Stage 5: summarise (GPU)

In [ ]:
!python run_qwen_video.py --root {ROOT}

In [ ]:
from google.colab import files
for f in ['summaries_video.csv', 'screen_text.csv', 'alignment.csv']:
    files.download(f'{ROOT}/outputs/video/{f}')